# 06 클러스터링 방법

**목적:** SBC(규칙 기반)와 대비할 **ML 비지도 클러스터**를 찾기 위해 임베딩×클러스터링 **24조합** 품질을 비교합니다.

| 임베딩 (6) | 클러스터링 (4) |
|-----------|---------------|
| PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST | KMeans, HAC, GMM, DBSCAN |

**선정 기준:** 실루엣 ↑ + Davies-Bouldin ↓ (낮을수록 좋음) 동시 순위, K=4만 후보  
**산출물:** `ml_cluster_type_family.parquet`


### ⓪ 데이터 준비: 시계열 행렬

165개 type×family 시계열을 **행=시계열, 열=주차(242)** 피벗 행렬로 변환합니다. 이후 6종 임베딩의 입력이 됩니다.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# 경로 설정
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, ML_CLUSTER
from utils.clustering_experiments import run_embedding_clustering_grid, select_best_combo, add_joint_rank

# 165 시계열 × 242주 피벗 행렬 생성
dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
pivot = dfw.pivot_table(index=['type', 'family'], columns='yearweek', values='sales', fill_value=0)
meta = pivot.index.to_frame(index=False)
X_raw = pivot.values.astype(float)
K = 4              # SBC 4분류와 동일한 클러스터 수
N_COMPONENTS = 10  # 임베딩 차원
print('series:', X_raw.shape[0], '| weeks:', X_raw.shape[1], '| K:', K)


series: 165 | weeks: 242 | K: 4


### ① 24조합 품질 실험

6 임베딩 × 4 클러스터링 = **24조합**을 일괄 실행하고, 실루엣·Davies-Bouldin(DB Index, **낮을수록 좋음**) 동시 순위표를 출력합니다.

In [2]:
# 6 임베딩 × 4 클러스터링 = 24조합 일괄 실행
quality_df, label_cache = run_embedding_clustering_grid(
    X_raw, k=K, n_components=N_COMPONENTS,
)
# 실루엣(↑좋음) + DB Index(↓좋음) 동시 순위
quality_ranked = add_joint_rank(quality_df, k=K)
print('=== K=4 조합: 실루엣·DB Index 동시 순위 (rank_score = sil_rank + db_rank, 낮을수록 우수) ===')
display(quality_ranked.round(4))


[embedding] PCA


  PCA+KMeans: silhouette=0.49409916553149, db=0.6896313697646943, n_clusters=4
  PCA+HAC: silhouette=0.8564530877738853, db=0.7436324163163133, n_clusters=4
  PCA+GMM: silhouette=0.2473123147734971, db=1.3003375769669379, n_clusters=4
  PCA+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] FastDTW


  FastDTW+KMeans: silhouette=0.8123490350068091, db=0.5188366934371917, n_clusters=4
  FastDTW+HAC: silhouette=0.8173942337871033, db=0.5003616061201555, n_clusters=4
  FastDTW+GMM: silhouette=0.8123490350068091, db=0.5188366934371917, n_clusters=4
  FastDTW+DBSCAN: silhouette=0.7339057572674218, db=0.27805358777026506, n_clusters=2
[embedding] AE


  AE+KMeans: silhouette=0.8399576544761658, db=0.6117928617810762, n_clusters=4
  AE+HAC: silhouette=0.80864417552948, db=0.5430995918381394, n_clusters=4
  AE+GMM: silhouette=0.7662065029144287, db=0.5916416973294485, n_clusters=4
  AE+DBSCAN: silhouette=0.8749735355377197, db=0.31629492317436214, n_clusters=3
[embedding] GAF-CNN


  GAF-CNN+KMeans: silhouette=0.49409916553149, db=0.6896313697646943, n_clusters=4
  GAF-CNN+HAC: silhouette=0.8564530877738853, db=0.7436324163163133, n_clusters=4
  GAF-CNN+GMM: silhouette=0.2473123147734971, db=1.3003375769669379, n_clusters=4
  GAF-CNN+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] TS2Vec
  TS2Vec+KMeans: silhouette=0.49409916553149, db=0.6896313697646943, n_clusters=4
  TS2Vec+HAC: silhouette=0.8564530877738853, db=0.7436324163163133, n_clusters=4


  TS2Vec+GMM: silhouette=0.2473123147734971, db=1.3003375769669379, n_clusters=4
  TS2Vec+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] PatchTST


  PatchTST+KMeans: silhouette=0.8961449861526489, db=0.32763345028460583, n_clusters=4
  PatchTST+HAC: silhouette=0.898070752620697, db=0.31871653438559594, n_clusters=4
  PatchTST+GMM: silhouette=0.5148922801017761, db=0.7065625567418743, n_clusters=4
  PatchTST+DBSCAN: silhouette=0.8649701476097107, db=0.17733506059984142, n_clusters=2
=== K=4 조합: 실루엣·DB Index 동시 순위 (rank_score = sil_rank + db_rank, 낮을수록 우수) ===


,method,embedding,clustering,n_clusters,silhouette,davies_bouldin,sil_rank,db_rank,rank_score
0,PatchTST+HAC,PatchTST,HAC,4,0.8981,0.3187,1.0,1.0,2.0
1,PatchTST+KMeans,PatchTST,KMeans,4,0.8961,0.3276,2.0,2.0,4.0
2,FastDTW+HAC,FastDTW,HAC,4,0.8174,0.5004,7.0,3.0,10.0
3,FastDTW+KMeans,FastDTW,KMeans,4,0.8123,0.5188,8.5,4.5,13.0
4,FastDTW+GMM,FastDTW,GMM,4,0.8123,0.5188,8.5,4.5,13.0
5,AE+KMeans,AE,KMeans,4,0.8400,0.6118,6.0,8.0,14.0
6,AE+HAC,AE,HAC,4,0.8086,0.5431,10.0,6.0,16.0
7,AE+GMM,AE,GMM,4,0.7662,0.5916,11.0,7.0,18.0
8,PCA+HAC,PCA,HAC,4,0.8565,0.7436,4.0,14.0,18.0
9,GAF-CNN+HAC,GAF-CNN,HAC,4,0.8565,0.7436,4.0,14.0,18.0


### ② 최적 조합 선정 및 저장

K=4 조합 중 **rank_score 최소** 조합의 클러스터 라벨을 `ml_cluster_type_family.parquet`에 저장합니다. PCA+KMeans와 비교 출력 포함.

In [3]:
# K=4 조합 중 rank_score 최소 → 최적 조합 선정
best = select_best_combo(quality_df, k=K)
print('최적 조합 (실루엣+DB 동시 순위, K=4):', best['method'])
print(f"rank_score={float(best['rank_score']):.1f}, n_clusters={int(best['n_clusters'])}, silhouette={float(best['silhouette']):.4f}, davies_bouldin={float(best['davies_bouldin']):.4f}")

# 참고: PCA+KMeans 비교 (하위권 조합)
pca_km = quality_df[quality_df['method'] == 'PCA+KMeans'].iloc[0]
print('비교 PCA+KMeans:', f"sil={float(pca_km['silhouette']):.4f}, db={float(pca_km['davies_bouldin']):.4f}, n_clusters={int(pca_km['n_clusters'])}")

# 최적 조합 라벨 저장 (1-based)
labels_best = label_cache[(best['embedding'], best['clustering'])]
out = meta.copy()
out['ML_CLUSTER'] = labels_best + 1
out['embedding_method'] = best['embedding']
out['clustering_method'] = best['clustering']
out.to_parquet(ML_CLUSTER, index=False)
quality_df.to_csv(DATA_PROCESSED / 'clustering_quality.csv', index=False)
quality_ranked.to_csv(DATA_PROCESSED / 'clustering_quality_ranked.csv', index=False)
print('저장:', ML_CLUSTER)
out.head()


최적 조합 (실루엣+DB 동시 순위, K=4): PatchTST+HAC
rank_score=2.0, n_clusters=4, silhouette=0.8981, davies_bouldin=0.3187
비교 PCA+KMeans: sil=0.4941, db=0.6896, n_clusters=4
저장: C:\Users\kjh\ai-retail-demandforecasting\data\processed\ml_cluster_type_family.parquet


,type,family,ML_CLUSTER,embedding_method,clustering_method
0,A,AUTOMOTIVE,1,PatchTST,HAC
1,A,BABY CARE,1,PatchTST,HAC
2,A,BEAUTY,1,PatchTST,HAC
3,A,BEVERAGES,2,PatchTST,HAC
4,A,BOOKS,1,PatchTST,HAC


## 분석 요약

### 선정 기준
- **실루엣 ↑** + **Davies-Bouldin ↓** 동시 반영 (`rank_score = sil_rank + db_rank`, **K=4 조합만**)
- DBSCAN(K≠4)은 제외 — SBC 4분류와 맞추기 위함

### K=4 조합 비교 (동시 순위)
| rank_score | 조합 | Silhouette | DB Index |
|------------|------|-----------|----------|
| **2** | **PatchTST+HAC** | **0.898** (실루엣 1위) | **0.319** (DB 1위) |
| 4 | PatchTST+KMeans | 0.896 | 0.328 |
| 10 | FastDTW+HAC | 0.817 | 0.500 |
| 14 | PCA+HAC | 0.856 | 0.744 (DB 나쁨) |
| **21** | **PCA+KMeans** | **0.494** | **0.690** |

→ **PatchTST+HAC**가 실루엣·DB Index **모두 K=4 조합 중 1위** (rank_score=2). PCA+HAC(0.856)는 실루엣 3위·DB 0.744로 동시 순위 14위. **PCA+KMeans**는 하위권(rank_score=21).

### 저장
- 최적: **PatchTST+HAC** → `ml_cluster_type_family.parquet`